# Experimento de Prueba — LightGBM Multivariado
Objetivo: verificar que el pipeline completo funciona de extremo a extremo.

**Flujo:**
1. Setup
2. Cargar y preparar datos
3. Verificar features
4. Visualizar CV splits
5. Correr CV con parámetros base
6. Loggear en MLflow
7. Evaluar en test set

## 1. Setup

In [0]:
%pip install lightgbm --quiet
dbutils.library.restartPython()

In [0]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('/Workspace/Repos/desareca/santiago-weather-forecast')

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from src.data.ingestion import load_from_delta_table
from src.data.preprocessing import build_features, train_test_split, get_feature_names
from src.models.lightgbm_model import LightGBMPredictor
from src.evaluation.metrics import evaluate_model
from src.evaluation.cross_validation import TimeSeriesSplit, run_cv
from src.utils.mlflow_utils import setup_experiment, log_cv_run, log_test_evaluation
from src.utils.config import *

setup_experiment()
print('✅ Setup completo')

## 2. Cargar y preparar datos

In [0]:
# Cargar datos crudos desde Delta
df_raw = load_from_delta_table('weather_raw', spark)

# Pipeline completo de features
df = build_features(df_raw)

# Split train / test
df_train, df_test = train_test_split(df)

print(f'\n📋 Columnas totales: {df.shape[1]}')
print(f'   Features X:      {len(get_feature_names(df))}')
print(f'   Target:          {TARGET}')

## 3. Verificar features

In [0]:
feature_names = get_feature_names(df)

print(f'Features ({len(feature_names)} total):')
for i, f in enumerate(feature_names):
    print(f'  {i+1:3d}. {f}')

print(f'\n🔍 NaN por columna en df_train:')
nan_counts = df_train[feature_names + [TARGET]].isna().sum()
nan_counts = nan_counts[nan_counts > 0]
if len(nan_counts) == 0:
    print('  ✅ Sin NaN')
else:
    print(nan_counts)

print(f'\n📊 Estadísticas del target ({TARGET}):')
print(df_train[TARGET].describe().round(3))

## 4. Visualizar CV splits

In [0]:
cv = TimeSeriesSplit(
    n_splits=N_SPLITS,
    test_size=TEST_SIZE,
    min_train_size=MIN_TRAIN_SIZE,
    strategy='sliding'
)
cv.visualize(df_train)

## 5. Cross-validation

In [0]:
# Parámetros base — punto de partida limpio
# No tocar hasta confirmar que el pipeline funciona end-to-end
BASE_PARAMS = {
    'objective':              'tweedie',
    'tweedie_variance_power': 1.5,
    'n_estimators':           300,
    'learning_rate':          0.05,
    'max_depth':              6,
    'num_leaves':             31,
    'min_child_samples':      20,
    'subsample':              0.8,
    'colsample_bytree':       0.8,
    'reg_alpha':              0.1,
    'reg_lambda':             1.0,
}

print('📋 Parámetros base:')
for k, v in BASE_PARAMS.items():
    print(f'  {k:30s}: {v}')

In [0]:
results_cv = run_cv(
    model_class=LightGBMPredictor,
    df=df_train,
    n_splits=N_SPLITS,
    test_size=TEST_SIZE,
    min_train_size=MIN_TRAIN_SIZE,
    strategy='sliding',
    **BASE_PARAMS
)

display(results_cv)

## 6. Loggear en MLflow

In [0]:
# Con sliding: entrenar con los datos del último fold
cv_temp = TimeSeriesSplit(
    n_splits=N_SPLITS,
    test_size=TEST_SIZE,
    min_train_size=MIN_TRAIN_SIZE,
    strategy='sliding'
)
last_train, _ = cv_temp.split(df_train)[-1]

model_final = LightGBMPredictor(**BASE_PARAMS)
model_final.fit(last_train)

print(f'📅 Modelo final entrenado con: '
      f'{last_train.index.min().date()} → {last_train.index.max().date()} '
      f'({len(last_train)} días)')

model_final.print_summary()

# Loggear en MLflow
run_id = log_cv_run(
    model=model_final,
    results_df=results_cv,
    model_params=BASE_PARAMS,
    run_name='LightGBM_baseline_v1',
    description='Baseline multivariado — todas las features, parámetros base',
    register_model=False,  # True solo cuando el modelo sea candidato a producción
)

print(f'\n🔗 Run ID: {run_id}')

## 7. Evaluación en test set

In [0]:
import matplotlib.pyplot as plt

# Predicciones en test
preds_test = model_final.predict(df_test)
y_test     = df_test[TARGET]

# Métricas
test_metrics = evaluate_model(y_test, preds_test, model_name='LightGBM_baseline_v1')

# Loggear métricas de test al run de CV
log_test_evaluation(
    run_id=run_id,
    test_metrics=test_metrics,
    description='Evaluación en test set holdout'
)

# Visualización
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('LightGBM Baseline — Evaluación en Test Set', fontsize=14, fontweight='bold')

# Panel 1: serie completa
axes[0].plot(y_test.index, y_test.values,
             label='Real', color='steelblue', alpha=0.8, linewidth=1.5)
axes[0].plot(preds_test.index, preds_test.values,
             label='Predicción', color='#e63946', alpha=0.8, linewidth=1.5)
axes[0].set_title('Precipitación real vs predicha — Test set completo')
axes[0].set_ylabel('mm')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Panel 2: zoom en meses de invierno (donde hay más señal)
mask_invierno = (y_test.index.month >= 5) & (y_test.index.month <= 9)
axes[1].plot(y_test[mask_invierno].index, y_test[mask_invierno].values,
             label='Real', color='steelblue', alpha=0.8, linewidth=1.5)
axes[1].plot(preds_test[mask_invierno].index, preds_test[mask_invierno].values,
             label='Predicción', color='#e63946', alpha=0.8, linewidth=1.5)
axes[1].set_title('Zoom: Mayo–Septiembre (temporada de lluvia)')
axes[1].set_ylabel('mm')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
importance = model_final.get_feature_importance(top_n=20)

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(
    importance['feature'][::-1],
    importance['importance_pct'][::-1],
    color='steelblue', edgecolor='white', alpha=0.85
)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_xlabel('Importancia (%)')
ax.set_title('Top 20 Features — LightGBM Baseline', fontsize=13, fontweight='bold')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

display(importance)